# AML Report 

### Imports

In [0]:
from pyspark.sql.functions import row_number, count, max, lit, col,crc32, concat_ws
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DecimalType
from datetime import datetime
from random import randint
from io import BytesIO
import pandas as pd
import os
import sys 


In [0]:
##aws_utils
import boto3
from botocore.exceptions import NoCredentialsError

from pyspark.dbutils import DBUtils
from pyspark.sql import SparkSession
 
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:  
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try: 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix


In [0]:
env = get_env_prefix()

### Insert existing AML Table and Calendar

We have ingested the tables from the on-premise environment for the first time that this runs in order to get historicity of AML data, meaning we retrieve all the records that have run before the implementation goes to the cloud. 
The **aml_report** table is the final table that will be used from the IT team to insert the new AML run data to the UI of the users.
The **execution calendar** table contains all the future dates that the AML is supposed to run.

In [0]:
aml_report_view = spark.sql(f""" SELECT * FROM `{env}-cdh-delivery`.aml.aml_report""")
aml_report_view.createOrReplaceTempView("aml_report_view")

execution_calendar = spark.sql(f""" SELECT * FROM `{env}-cdh-delivery`.aml.execution_calendar""")
execution_calendar.createOrReplaceTempView("execution_calendar")


### Generate aml_sch_id field for final AML table

This field is a randomly generated code used in the on-premise implementation. Therefore we keep this and generate it from 1000, starting from the on-premise codes.

In [0]:
%python
## Generate new aml_sch_id (random 4 digit number) that does not already exist in aml_report
existing_ids = [row['aml_sch_id'] for row in spark.sql("SELECT DISTINCT aml_sch_id FROM aml_report_view").collect()]

while True:
    new_aml_sch_id = randint(1000, 9999)
    if new_aml_sch_id not in existing_ids:
        break

### Check if AML can run for the current month and assign Businessdate

In [0]:
import calendar as cal_mod
from datetime import date

#Read widget values
_force_rerun = dbutils.widgets.get("force_rerun") == "true"
_rerun_month = dbutils.widgets.get("rerun_month")  # format: YYYY-MM

if _force_rerun and _rerun_month:
    # FORCE RERUN MODE — derive dates from widget
    
    _year, _month = int(_rerun_month.split('-')[0]), int(_rerun_month.split('-')[1])
    DateFrom = date(_year, _month, 1)
    DateTo = date(_year, _month, cal_mod.monthrange(_year, _month)[1])
    AMLYear = date(_year, 1, 1)
    AMLEndOfYear = date(_year, 12, 31)

    # Previous month last day
    if _month == 1:
        PreviousMonthDateTo = date(_year - 1, 12, 31)
    else:
        PreviousMonthDateTo = date(_year, _month - 1, cal_mod.monthrange(_year, _month - 1)[1])

    # ExecutionDate = 1st of next month (for correct JanuaryFlag downstream)
    if _month == 12:
        ExecutionDate = date(_year + 1, 1, 1)
    else:
        ExecutionDate = date(_year, _month + 1, 1)

    OUTPUT_TABLE = f"`{env}-cdh-delivery`.aml.aml_report"

    # Read existing AML data (for historical UNION downstream)
    
    aml_report_view = spark.sql(f"""
        SELECT * FROM `{env}-cdh-delivery`.aml.aml_report
        WHERE date_to < '{DateFrom}'
    """)
    aml_report_view.createOrReplaceTempView("aml_report_view")

    # Read execution calendar
    execution_calendar = spark.sql(f"SELECT * FROM `{env}-cdh-delivery`.aml.execution_calendar")
    execution_calendar.createOrReplaceTempView("execution_calendar")

    # Generate next aml_sch_id
    new_aml_sch_id = spark.sql("SELECT COALESCE(MAX(id), 0) + 1 FROM execution_calendar").first()[0]

    AMLExecutionFlag = 1

    # Create AmlExecutionCalendar temp view (needed by downstream cells)
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW InitialCalendar AS
        SELECT {new_aml_sch_id} AS id, {_year} AS ExecYear, 'M' AS ExecType, {_month} AS ExecPart,
            DATE '{DateFrom}' AS DateFrom, DATE '{DateTo}' AS DateTo,
            DATE '{PreviousMonthDateTo}' AS PreviousMonthLastDate,
            DATE '{AMLYear}' AS AMLYear, DATE '{AMLEndOfYear}' AS AMLEndOfYear,
            {new_aml_sch_id} AS aml_sch_id, NULL AS ActualExecution,
            DATE '{ExecutionDate}' AS ExecutionDate
    """)
    spark.sql("CREATE OR REPLACE TEMP VIEW AmlExecutionCalendar AS SELECT * FROM InitialCalendar")

    print(f"FORCE RERUN for {_rerun_month}")
    print(f"Period: {DateFrom} -> {DateTo} | AMLYear: {AMLYear} | JanuaryFlag: {'1 (Jan data)' if _month == 1 else '0'}")
    print(f"WRITE: {OUTPUT_TABLE}")
    print(f"aml_report_view: {len(aml_report_view.columns)} fields")

else:
    # NORMAL MODE — use execution_calendar
    
    InitialCalendar = spark.sql(f"""
        SELECT DISTINCT a.id,
            CalendarYear AS ExecYear,  
            'M' AS ExecType,
            CalendarMonth AS ExecPart,
            FirstDateOfMonth AS DateFrom,
            LastDateOfMonth AS DateTo,
            PreviousMonthLastDate,
            TRY_CAST(CONCAT(CalendarYear, '-01-01') AS DATE) AS AMLYear,
            TRY_CAST(CONCAT(CalendarYear, '-12-31') AS DATE) AS AMLEndOfYear,
            {new_aml_sch_id} AS aml_sch_id,
            a.ActualExecution,
            a.ExecPart,
            a.ExecType, 
            a.ExecutionDate
        FROM `{env}-cdh`.reference.ref_calendar i
        JOIN execution_calendar a
        ON i.FirstDateOfMonth = a.DateFrom
        WHERE CalendarYear = case when month(a.ExecutionDate) = '01' THEN YEAR(CURRENT_DATE())-1 ELSE YEAR(CURRENT_DATE()) END
        AND MONTH(CalendarDate) = case when month(a.ExecutionDate) = '01'  THEN '12' ELSE MONTH(DATE(getdate())) - 1 END
    """)

    InitialCalendar.createOrReplaceTempView("InitialCalendar")

    AmlExecutionCalendar = spark.sql(f""" 
        SELECT * 
        FROM InitialCalendar
        WHERE (ActualExecution IS NULL OR ActualExecution == "NULL")
        AND ExecutionDate <  TRY_CAST(getdate() as date)
    """)
    AmlExecutionCalendar.createOrReplaceTempView("AmlExecutionCalendar")

    # If df is not empty then we can execute the AML Process
    if AmlExecutionCalendar.count() == 0:
        AMLExecutionFlag = 0
        print("Error: AML Cannot Run with today's flow. Stopping execution.")
    
    else:
        AMLExecutionFlag = 1
        if AmlExecutionCalendar.count() > 0:
            calendar_row = AmlExecutionCalendar.first()
        else:
            calendar_row = InitialCalendar.first()
        print("AML can run for", calendar_row['DateFrom'], calendar_row['DateTo'])
    
        DateFrom            = calendar_row["DateFrom"]
        DateTo              = calendar_row["DateTo"]
        AMLYear             = calendar_row["AMLYear"]
        AMLEndOfYear        = calendar_row["AMLEndOfYear"]
        PreviousMonthDateTo = calendar_row["PreviousMonthLastDate"]
        ExecutionDate       = calendar_row["ExecutionDate"]

        OUTPUT_TABLE = f"`{env}-cdh-delivery`.aml.aml_report"
    
        print(f"DateFrom: {DateFrom}")
        print(f"DateTo: {DateTo}")
        print(f"AMLYear: {AMLYear}")
        print(f"AMLEndOfYear: {AMLEndOfYear}")
        print(f"PreviousMonthDateTo: {PreviousMonthDateTo}")
        print(f"ExecutionDate: {ExecutionDate}")


if AMLExecutionFlag == 1:
    # Scoped BD: max within processing month 
    Businessdate = spark.sql(f"""
        SELECT MAX(BusinessDate) AS BusinessDate
        FROM `{env}-cdh`.derivative.DCollection
        WHERE CAST(BusinessDate AS DATE) BETWEEN '{DateFrom}' AND '{DateTo}'
    """).collect()[0]["BusinessDate"]
    if Businessdate is None:
        raise ValueError(f"No BusinessDate in DCollection for {DateFrom} → {DateTo}. Check data availability.")

    # Global BD: latest available 
    Businessdate_coll = spark.sql(f"""
        SELECT MAX(BusinessDate) AS BusinessDate
        FROM `{env}-cdh`.derivative.DCollection
    """).collect()[0]["BusinessDate"]

    print(f"Businessdate:      {Businessdate} (scoped to {DateFrom} → {DateTo})")
    print(f"Businessdate_coll: {Businessdate_coll} (global max — for DCollection/DExposure/TotalPayments)")

## Create AmlPerimeter

### Portfolios Perimeter

Create the perimeter of the Portfolios for which we can produce the AML. The ones that are excludes are **Artemis, Core Retail, Core Wholesale, Cosmos Common Customers, Galaxy Common Customers, Hermes Common Customers.**

In [0]:
if AMLExecutionFlag == 1:
    PortfolioPerimeter = spark.sql(f"""                    
                                SELECT p.portfolioname, Prefix, unit_code_profits
                                FROM `{env}-cdh`.primaryactive.portfolio p
                                WHERE  unit_code_profits IS NOT NULL 
                                AND p.PortfolioName NOT LIKE '%common%' 
                                AND p.PortfolioName NOT LIKE 'N/A' 
                                AND p.IsBankingFlag = 'FALSE'
                            """)

    PortfolioPerimeter.createOrReplaceTempView("PortfolioPerimeter")

### Restructuring Perimeter

In [0]:
if AMLExecutionFlag == 1 :
    Restructuring = spark.sql(f""" SELECT NewExposurekey, cdi, RestructuringAmount, BusinessDate
                                    FROM `{env}-cdh`.primaryactive.restructuring   
                                    WHERE BusinessDate = '{Businessdate}'
                              """)
    Restructuring.createOrReplaceTempView("Restructuring")

### Payment Schedule Perimeter

In [0]:
if AMLExecutionFlag == 1 :
    PaymentSchedule = spark.sql(f""" SELECT ExposureKey, InstallmentAmount, InstallmentAmountPaid, InstallmentDate, InstallmentType, BusinessDate
                                    FROM `{env}-cdh`.`primary`.paymentschedule   
                                    WHERE BusinessDate = '{Businessdate}'
                              """)
    PaymentSchedule.createOrReplaceTempView("PaymentSchedule")


### DExposure Perimeter

In [0]:
if AMLExecutionFlag == 1 :
    
    DExposure = spark.sql(f"""        
                            SELECT  DISTINCT act.exposurekey,
                                             l.CDI, 
                                             act.BusinessDate
                                    FROM  `{env}-cdh`.derivativeactive.Dexposure l  
                                    join `{env}-cdh`.primary.accounttransaction act
                                    on act.exposurekey = l.exposurekey
                                    WHERE act.BusinessDate = '{Businessdate}'
                                                                   """)

    DExposure.createOrReplaceTempView("DExposure")

### High Risk Exposures
Flag exposures as HR if ANY related party (Primary Owner / Guarantor / Co-Owner) matches AML Tool criteria from `high_risk_customers_export`:  
- **HR Country**: `Nationality_RiskScore = 'HIGH'` OR `Citizenship_RiskScore = 'HIGH'`  
- **Sanctions**: `SanctionReviewedMatched = true` OR `CautionReviewedMatched = true`  

HR exposures follow the **€3k threshold / €3k step** rule (no exclusions).

In [0]:
if AMLExecutionFlag == 1:
    # Flag exposures as HR via AML Tool (high_risk_customers_export): HR Country or Sanctions match
    HighRiskExposures = spark.sql(f"""
        SELECT e.ExposureKey,
               1 AS IsHighRiskExposure
        FROM `{env}-cdh`.primaryactive.exposure e
        JOIN `{env}-cdh`.derivative.Dparty dp
          ON dp.CDI = e.CDI
          AND dp.BusinessDate = '{Businessdate}'
        JOIN (
            SELECT TaxRegistrationNumber
            FROM `{env}-cdh-delivery`.aml.high_risk_customers_export
            WHERE export_date = (SELECT MAX(export_date) FROM `{env}-cdh-delivery`.aml.high_risk_customers_export)
              AND (
                  Nationality_RiskScore = 'HIGH' OR Citizenship_RiskScore = 'HIGH'
                  OR SanctionReviewedMatched = true OR CautionReviewedMatched = true
              )
            GROUP BY TaxRegistrationNumber
        ) hrc ON hrc.TaxRegistrationNumber = dp.TaxRegistrationNumber
        WHERE e.BusinessDate = '{Businessdate}'
          AND e.PartyRelationshipType IN ('Primary Owner', 'Guarantor', 'Co-Owner')
        GROUP BY e.ExposureKey
    """)
    HighRiskExposures.createOrReplaceTempView("HighRiskExposures")
    print(f"HighRiskExposures: {HighRiskExposures.count():,} exposures flagged as High Risk")

### Collections Perimeter

In [0]:
if AMLExecutionFlag == 1:
        DCollection = spark.sql(f"""        
                            SELECT  DISTINCT 
                                    DCollectionKey
                                    , PrimaryOwnerCDI
                                    , ExposureKey
                                    , SourceExposureKey
                                    , cast(ActualDate as date) PayDateRd
                                    , Amount                   PaymentAmount
                                    , PortfolioName
                                    , if(PaymentType ='N/A','cash',PaymentType) PaymentType 
                                    , PaymentTypeGroup
                                    , PortfolioKey
                                    , aml_sch_id
                                    , PayerCDI
                                    , PaymentFileChannel
                                    FROM `{env}-cdh`.derivative.dcollection  
                                    JOIN AMLExecutionCalendar ex 
                                    on cast(ActualDate as date) between AMLYear AND DateTo
                                    WHERE FinalReason = 'Matched'
                                        AND  (ltrim(rtrim(PaymentTypeGroup)) not in 
                                                (SELECT Description FROM `{env}-cdh-delivery`.aml.paymenttype_auctions
                                                WHERE FieldName = 'PaymentTypeGroup') 
                                        OR PaymentTypeGroup is null)
                                        AND (ltrim(rtrim(PaymentType)) not in 
                                                (SELECT Description FROM `{env}-cdh-delivery`.aml.paymenttype_auctions WHERE FieldName = 'PaymentType') 
                                                OR PaymentType is null)
                                        AND ltrim(rtrim(PaymentType)) not like '%Collection through%'
                                        AND ltrim(rtrim(PaymentType)) not like '%Gross Proceeds%'
                                        AND ltrim(rtrim(PaymentType)) not like '%REO%'
                                        AND ltrim(rtrim(PaymentType)) not like '%Non-RE Collateral Auction Proceeds%'  
                                        AND BusinessDate = '{Businessdate_coll}'     
                                """)       

        DCollection.createOrReplaceTempView("DCollection")

### Declared Income

Find the **Declared Income** from the interview for each customer that has been stated for **the latest PositionDate & CurrentAssignmentDate.** 

In [0]:
if AMLExecutionFlag == 1:
    Application = spark.sql(f"""        
                                    SELECT id, CustomerCode, CurrentStepAssignmentDate,applicationstatusdescription
                                    FROM `{env}-cdh`.landed.fldt_application 
                                    WHERE BusinessDate = '{Businessdate}'""")
    Application.createOrReplaceTempView("Application")

In [0]:
if AMLExecutionFlag == 1:
    Customer = spark.sql(f"""        
                                    SELECT id, CustomerCode, ApplicationId, PositionDate
                                    FROM `{env}-cdh`.landed.fldt_customer 
                                    WHERE BusinessDate = '{Businessdate}'""")
    Customer.createOrReplaceTempView("Customer")

In [0]:
if AMLExecutionFlag == 1:
    MasterInterview = spark.sql(f"""        
                                    SELECT id, CustomerId
                                    FROM `{env}-cdh`.landed.fldt_masterinterview  
                                    WHERE BusinessDate = '{Businessdate}'""")
    MasterInterview.createOrReplaceTempView("MasterInterview")

In [0]:
if AMLExecutionFlag == 1:
    FinancialAssessmentInterview = spark.sql(f"""        
                                    SELECT masterinterviewid, DeclaredIncome
                                    FROM `{env}-cdh`.landed.fldt_financialassessmentinterview  
                                    WHERE BusinessDate = '{Businessdate}'""")
    FinancialAssessmentInterview.createOrReplaceTempView("FinancialAssessmentInterview")

In [0]:
if AMLExecutionFlag == 1:
    latestCustomerIncome = spark.sql(f""" 
        SELECT DISTINCT a.CustomerCode,
               f.DeclaredIncome,
               ROW_NUMBER() OVER (
                   PARTITION BY prt.TaxRegistrationNumber
                   ORDER BY c.PositionDate DESC, a.CurrentStepAssignmentDate DESC
               ) AS rn
        FROM Customer c
        JOIN Application a
          ON c.applicationid = a.id
        JOIN MasterInterview m
          ON m.CustomerId = c.id
        JOIN FinancialAssessmentInterview f
          ON f.masterinterviewid = m.id
        JOIN `{env}-cdh`.derivative.Dparty prt
        ON c.CustomerCode = prt.CDI  
        WHERE a.applicationstatusdescription <> 'Canceled'
    """)
    latestCustomerIncome.createOrReplaceTempView("latestCustomerIncome")


In [0]:
if AMLExecutionFlag == 1:
    FinalIncomePerimeter = spark.sql(f"""   
                SELECT DISTINCT
                    CustomerCode,
                    DeclaredIncome
                FROM latestCustomerIncome
                WHERE rn = 1""")
    FinalIncomePerimeter.createOrReplaceTempView("FinalIncomePerimeter")

### DParty Perimeter

In [0]:
if AMLExecutionFlag == 1:
    DParty = spark.sql(f"""        
        SELECT  DISTINCT
               prt.cdi,
               prt.PartyType AS PartyType,
               prt.TaxRegistrationNumber AS TaxNumber,
               IFNULL(prt.FirstNameGR, '') AS FirstName,
               IFNULL(prt.LastNameGR, '') AS LastName,
               IFNULL(prt.FirstFatherNameGR, '') AS FatherName,
               IFNULL(prt.Nationality, '') AS Nationality,
               prt.BirthDate,
               IFNULL(prt.Occupation, '') AS Occupation,
               prt.PortfolioName,
               CASE WHEN prt.PartyType = 'Organization' THEN 'Yes' ELSE 'No' END AS LegalEntity,
               CASE WHEN prt.RiskDescription = 'Low' THEN 1 ELSE 0 END AS IsLowRiskFlag, 
               fi.DeclaredIncome,
               prt.GroupName,
               prt.DeceasedFlag,
               prt.PartyStatus,
               NULLIF(prt.PartyStatusDate, '1900-01-01') AS PartyStatusDate
               ,CASE WHEN prt.EnterpriseEndDate IN (DATE('1753-01-01'), DATE('0001-01-01'), DATE('9999-12-31'))
                     THEN NULL ELSE prt.EnterpriseEndDate END AS LegitimizationExpiredDate
        FROM `{env}-cdh`.derivative.Dparty prt  
        JOIN PortfolioPerimeter pp
          ON prt.PortfolioName = pp.portfolioname
        LEFT JOIN FinalIncomePerimeter fi
          ON fi.CustomerCode = prt.cdi
          AND fi.DeclaredIncome > 0
        WHERE prt.TaxRegistrationNumber IS NOT NULL 
          AND prt.BusinessDate = '{Businessdate}'
    """)
    DParty.createOrReplaceTempView("DParty")


In [0]:
if AMLExecutionFlag == 1:
    TotalPaymentsPerimeter = spark.sql(f"""
        SELECT
            prt.TaxNumber,
            SUM(CAST(dc.Amount AS DECIMAL(18,2)))                                              AS total_payments_ever,
            SUM(CASE WHEN YEAR(CAST(dc.ActualDate AS DATE)) = YEAR(DATE('{AMLYear}'))
                     THEN CAST(dc.Amount AS DECIMAL(18,2)) ELSE 0 END)                         AS total_payments_current_year
        FROM `{env}-cdh`.derivative.dcollection dc
        JOIN DParty prt ON prt.CDI = dc.PrimaryOwnerCDI
        WHERE dc.FinalReason = 'Matched'
          AND dc.BusinessDate = '{Businessdate_coll}'
          AND (ltrim(rtrim(dc.PaymentTypeGroup)) NOT IN (
                    SELECT Description FROM `{env}-cdh-delivery`.aml.paymenttype_auctions
                    WHERE FieldName = 'PaymentTypeGroup')
               OR dc.PaymentTypeGroup IS NULL)
          AND (ltrim(rtrim(dc.PaymentType)) NOT IN (
                    SELECT Description FROM `{env}-cdh-delivery`.aml.paymenttype_auctions
                    WHERE FieldName = 'PaymentType')
               OR dc.PaymentType IS NULL)
          AND ltrim(rtrim(dc.PaymentType)) NOT LIKE '%Collection through%'
          AND ltrim(rtrim(dc.PaymentType)) NOT LIKE '%Gross Proceeds%'
          AND ltrim(rtrim(dc.PaymentType)) NOT LIKE '%REO%'
          AND ltrim(rtrim(dc.PaymentType)) NOT LIKE '%Non-RE Collateral Auction Proceeds%'
        GROUP BY prt.TaxNumber
    """)
    TotalPaymentsPerimeter.createOrReplaceTempView("TotalPaymentsPerimeter")
    row_count = TotalPaymentsPerimeter.count()
    print(f"TotalPaymentsPerimeter created with {row_count:,} rows")

### Final amlPerimeter

This is the final amlPerimeter view that we use multiple times in the code. That's the reason why we cache it since it is used all throughout the notebook. We get all the data from all the tables stated above and **keep only the collections that refer to payments from the start of the year until the BusinessDate used to run the notebook.**

In [0]:
if AMLExecutionFlag ==1:
        amlPerimeter = spark.sql(f"""        
                            SELECT  DISTINCT prt.CDI
                                    , l.ExposureKey
                                    , coll.PayDateRd
                                    , coll.PaymentAmount
                                    , prt.PartyType
                                    , prt.PortfolioName 
                                    , coll.PaymentType 
                                    , prt.TaxNumber
                                    , prt.FirstName
                                    , prt.LastName
                                    , prt.FatherName
                                    , prt.Nationality
                                    , prt.BirthDate
                                    , prt.Occupation
                                    , prt.LegalEntity
                                    , prt.IsLowRiskFlag
                                    , prt.DeclaredIncome
                                    , coll.aml_sch_id
                                    , prt.GroupName
                                    , prt.DeceasedFlag
                                    , prt.PartyStatus
                                    , prt.PartyStatusDate
                                    , coll.PayerCDI
                                    , coll.PaymentFileChannel
                                    , prt.LegitimizationExpiredDate
                                    , COALESCE(hr.IsHighRiskExposure, 0) AS IsHighRiskExposure
                                    FROM  DCollection coll 
                                    JOIN PortfolioPerimeter por
                                    ON por.unit_code_profits=coll.portfolioKey         
                                    JOIN DExposure l    
                                    ON l.ExposureKey = coll.ExposureKey
                                    JOIN DParty prt  
                                    ON prt.cdi = l.cdi
                                    LEFT JOIN HighRiskExposures hr
                                    ON hr.ExposureKey = l.ExposureKey
                                                                      
                                     """)
        amlPerimeter.cache()
        row_count = amlPerimeter.count()
        hr_count = amlPerimeter.filter("IsHighRiskExposure = 1").count()
        print(f"amlPerimeter created with {row_count:,} rows ({hr_count:,} High Risk)")
                
        # Keep full perimeter for later use (HR + Standard)
        amlPerimeter.createOrReplaceTempView("amlPerimeterFull")

        # Override amlPerimeter view with Standard-only for existing downstream logic
        spark.sql("CREATE OR REPLACE TEMP VIEW amlPerimeter AS SELECT * FROM amlPerimeterFull WHERE IsHighRiskExposure = 0")
        print(f"amlPerimeter view overridden to Standard-only ({row_count - hr_count:,} rows)")

### Customers with multiple Portfolios

For each VAT Number where it we have collection data in different portfolios, we use it later on to concat all the different portfolios using a comma delimeter between each portfolio.

In [0]:
if AMLExecutionFlag==1:
     ## All Customers from the current perimeter with at least 2 portfolios per TaxNumber
     amlCommonCustomersDuplicates = spark.sql(f""" SELECT 
                                              TaxNumber,  
                                              COUNT(DISTINCT PortfolioName)
                                    FROM amlPerimeter
                                    GROUP BY TaxNumber
                                    HAVING COUNT(DISTINCT PortfolioName)>1
                               """)
     amlCommonCustomersDuplicates.createOrReplaceTempView("amlCommonCustomersDuplicates")

     amlCommonCustomers = spark.sql(f""" SELECT DISTINCT a.TaxNumber, a.PortfolioName
                                    FROM amlCommonCustomersDuplicates cc
                                    join amlPerimeter a on a.TaxNumber = cc.TaxNumber
                               """)
     amlCommonCustomers.createOrReplaceTempView("amlCommonCustomers")

# Functions

### Duplicate Payment Types

Find all VAT Numbers with duplicate Payment Types. If one VAT has two or more Payment Types, we use NULL as the final Type in the AML Table.

In [0]:
def find_duplicate_payment_type(CustomersPerimeter, datefrom: str, dateto: str, view_name: str, spark):

    CustomersPerimeter.createOrReplaceTempView("CustomersPerimeterInput")
    query = f"""
        CREATE OR REPLACE TEMP VIEW {view_name} AS
        SELECT a.TaxNumber,
               COUNT(DISTINCT a.paymenttype) AS countPaymentType
        FROM amlPerimeter a
        join CustomersPerimeterInput b on a.Taxnumber = b.Taxnumber
        GROUP BY a.TaxNumber
        HAVING COUNT(DISTINCT a.paymenttype) > 1

    """
    spark.sql(query)


### Minimum and Maximum PayDate 

We find the first payment of each VAT Number and the Last one in order to **use the first day of the month of the Min Payment** and **the last day of the month of the Max Payment.** For example, the first date was '2025-03-04' and the last '2025-09-28'. Therefore, the dates we will use in the final AML are date_from = '2025-03-01' and date_to = '2025-09-30'.

In [0]:
def calculate_aml_period(CustomersPerimeter,view_name: str, spark):
    CustomersPerimeter.createOrReplaceTempView("CustomersPerimeterInput")

    query = f"""
        CREATE OR REPLACE TEMP VIEW {view_name} AS
        SELECT DISTINCT 23 AS Ruleid,  
								b.taxnumber, 
								b.PaymentAmount, 
								NumOfPayment, 
								NumOfPortfolio, 
								date(concat(year(b.minPayDate), '-', lpad(month(b.minPayDate), 2, '0'), '-01')) minPaydate,
								date_sub(add_months(date(concat(year(b.maxPayDate), '-', lpad(month(b.maxPayDate), 2, '0'), '-01')), 1), 1) maxPaydate,
								c.PortfolioName,
								c.PaymentType
								FROM CustomersPerimeterInput b
								join amlPerimeter c on b.taxnumber = c.TaxNumber							
								
    """
    spark.sql(query)



### Ytd Payments 

We calculate the YearToDate collections for each Vat Number for the specific calculation time period. For example, if we need YtD we used date_from = '2025-01-01' and date_to = last day of current month. Otherwise, the date_from = first date of current month if we are calculating MtD.

In [0]:
from datetime import date
def calculate_YtD_Payments(CustomersPerimeter, datefrom: date, dateto: date, view_name: str, spark):
    CustomersPerimeter.createOrReplaceTempView("CustomersPerimeter")

    query = f"""
        CREATE OR REPLACE TEMP VIEW {view_name} AS
                                            SELECT   TaxNumber
		  											, sum(PaymentAmount) PaymentAmount
                                                    , count(1) NumOfPayment
                                                    , count(distinct PortfolioName) NumOfPortfolio
										            , min(paydaterd) minPayDate  
										            , max(paydaterd) maxPayDate 											
                                            from CustomersPerimeter
											where PayDateRd   between  DATE('{datefrom.isoformat()}') AND DATE('{dateto.isoformat()}')     
											group by TaxNumber
											HAVING sum(PaymentAmount) >= 15000											
    """
    spark.sql(query)


 

### OnceOff Payments

We include all payments that have been paid once-off before they were due and that are restructured even if they are excluded from the other rules: 35% family income & low risk

In [0]:
def calculate_OnceOff_Payments(view_name: str, perimeter: str, spark):

    query = f"""
        CREATE OR REPLACE TEMP VIEW {view_name} AS
                                           
                            select NewExposureKey,TaxNumber,PaymentAmount,sum(round(InstallmentAmount,2)) InstallmentAmount
                            from (
                                select DISTINCT a.NewExposureKey,d.TaxNumber, p.InstallmentAmount, p.InstallmentDate, aml.PaymentAmount as  PaymentAmount                
                                from    Restructuring a
                                inner join DParty d
                                on d.cdi = a.cdi    
                                inner join {perimeter} aml
                                on d.TaxNumber = aml.TaxNumber   
                                inner join PaymentSchedule p
                                on        p.ExposureKey=a.NewExposureKey
                                and       p.BusinessDate=a.BusinessDate
                                and       p.InstallmentType='Installment'
                                and       p.InstallmentAmountPaid >0
                                inner join InitialCalendar i
                                on p.InstallmentDate between AMLYear and AMLEndOfYear
                                where     a.BusinessDate='{Businessdate}'
                            )a
                            WHERE a.PaymentAmount <= 50000
                            group  by NewExposureKey, TaxNumber, PaymentAmount
                            HAVING a.PaymentAmount > sum(round(InstallmentAmount,2))  
                           	
    """
    spark.sql(query)


 

### Low Risk Customers with Sum of Payments below 50k 

For this rule we exclude from AML all the VAT Numbers that refer to 'Low Risk' customers with Payments over 15k and under 50k (New Rule https://cepalhellas.atlassian.net/browse/ASK-3466)

In [0]:
def calculate_LowRisk_Customers(view_name: str, perimeter: str, spark):

    query = f"""
        CREATE OR REPLACE TEMP VIEW {view_name} AS
                                            SELECT DISTINCT a.TaxNumber
		  											        ,  a.PaymentAmount
											from {perimeter} a
                                            join amlPerimeter b
                                            on a.TaxNumber = b.TaxNumber
											where a.PaymentAmount between 15000 and 50000 
                                            AND b.IsLowRiskFlag = 1						
    """
    spark.sql(query)

### Family Income Rule

We exclude all VAT Numbers where the sum of Payment Amount is less than 35% of the Declared Income and the is also less than 100k (New Rules https://cepalhellas.atlassian.net/browse/ASK-3466)

In [0]:
def calculate_FamilyIncome_35percent_rule(view_name: str, perimeter: str, spark):

    query = f"""
        CREATE OR REPLACE TEMP VIEW {view_name} AS
                                            SELECT DISTINCT a.TaxNumber
		  											        ,  a.PaymentAmount
                                                            , b.DeclaredIncome     
											from {perimeter} a
                                            join amlPerimeter b
                                            on a.TaxNumber = b.TaxNumber
											where a.PaymentAmount between 15000 AND 100000
                                            and (b.DeclaredIncome * 0.35 >= a.PaymentAmount)							
    """
    spark.sql(query) 

### High Risk Alert Calculation — Step-by-Step Overview

For High Risk exposures the alert threshold is **€3,000** (instead of the standard €15k / €50k), with alerts every **€3,000 step**. No Low Risk / Family Income / OnceOff exclusions apply to HR customers.


In [0]:
def calculate_HighRisk_YtD(datefrom, dateto, view_name: str, spark):
    """Calculate YtD payments for High Risk perimeter from amlPerimeterFull where IsHighRiskExposure=1."""
    query = f"""
        CREATE OR REPLACE TEMP VIEW {view_name} AS
        SELECT   TaxNumber
                , SUM(PaymentAmount) AS PaymentAmount
                , COUNT(PayDateRd) AS NumOfPayment  -- count individual payment transactions
                , COUNT(DISTINCT PortfolioName) AS NumOfPortfolio
                , MIN(PayDateRd) AS minPayDate
                , MAX(PayDateRd) AS maxPayDate
        FROM amlPerimeterFull
        WHERE IsHighRiskExposure = 1
          AND PayDateRd BETWEEN '{datefrom}' AND '{dateto}'
        GROUP BY TaxNumber
        HAVING SUM(PaymentAmount) >= 3000
    """
    spark.sql(query)

## Aggregation of Portfolios - Multiple Portfolio per TaxNumber

When one VAT Number has payments in different portfolios for the specific calculation period then we agreggate the portfolios using comma delimiters in between them (part of ASK-4550).

In [0]:
def update_portfolio_of_payment(view_name: str):
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {view_name} AS
        SELECT 
            a.taxnumber,
            CONCAT_WS(',', COLLECT_SET(a.PortfolioName)) AS portfolio
        FROM amlPerimeter a
        INNER JOIN FinalTableRank_withchecksum r 
            ON a.taxnumber = r.Tax_number
            AND a.PayDateRd >= r.Datefrom
        GROUP BY a.taxnumber
    """)

## Aggregation of CDIs - Multiple CDIs per TaxNumber

When one VAT Number has payments for the specific calculation period in accounts of more than one portfolio then we agreggate the CDIs using comma delimiters in between them (part of ASK-4550).

In [0]:
def update_cdi(view_name: str):
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {view_name} AS
        SELECT 
            a.taxnumber,
            CONCAT_WS(',', COLLECT_SET(a.CDI)) AS CDI
        FROM amlPerimeter a
        INNER JOIN FinalTableRank_withchecksum r 
            ON a.taxnumber = r.Tax_number
            AND a.PayDateRd >= r.Datefrom
        GROUP BY a.taxnumber
    """)

## Flag for January Run

**There are different (simpler) rules for the January Run and the Run for the rest of the months.** Therefore, we create a flag to decide the calculations that will be used depending on the month (January or not).

In [0]:

## Check if we are currently in the January Run or not
if AMLExecutionFlag==1:
    JanuaryFlag = spark.sql(f"""                    
                        select case when month(ExecutionDate)-1 = 1 then 1 else 0 end JanuaryFlag from AMLExecutionCalendar
                       
                    """).first()[0]

    print(JanuaryFlag)

### January Rule Calculations

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 1:
  
    # Existing Customers
    calculate_YtD_Payments(amlPerimeter,DateFrom, DateTo, "JanuaryCustomers_df", spark)
    JanuaryCustomersYtD= spark.sql("SELECT * FROM JanuaryCustomers_df")
    JanuaryCustomersYtD.createOrReplaceTempView("JanuaryCustomersYtD")

    # Low risk Customers with less than 50k sum of payments
    calculate_LowRisk_Customers("JanuaryCustomersLowRisk_df", "JanuaryCustomersYtD", spark)
    JanuaryCustomersLowRisk= spark.sql("SELECT * FROM JanuaryCustomersLowRisk_df")
    JanuaryCustomersLowRisk.createOrReplaceTempView("JanuaryCustomersLowRisk")

    # Final January Customers Perimeter
    JanuaryCustomers = spark.sql(f""" 
                                 SELECT a.* 
                                 FROM JanuaryCustomersYtD a
                                 LEFT JOIN JanuaryCustomersLowRisk b ON a.TaxNumber = b.TaxNumber
                                 WHERE b.TaxNumber IS NULL
                                 """)
    JanuaryCustomers.createOrReplaceTempView("JanuaryCustomers")

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 1:
  
    # Existing Customers
    calculate_YtD_Payments(amlPerimeter,DateFrom, DateTo, "JanuaryCustomers_df", spark)
    JanuaryCustomersYtD= spark.sql("SELECT * FROM JanuaryCustomers_df")
    JanuaryCustomersYtD.createOrReplaceTempView("JanuaryCustomersYtD")

    # Low risk Customers with less than 50k sum of payments
    calculate_LowRisk_Customers("JanuaryCustomersLowRisk_df", "JanuaryCustomersYtD", spark)
    JanuaryCustomersLowRisk= spark.sql("SELECT * FROM JanuaryCustomersLowRisk_df")
    JanuaryCustomersLowRisk.createOrReplaceTempView("JanuaryCustomersLowRisk")

    # Low risk Customers with less than 50k sum of payments
    calculate_FamilyIncome_35percent_rule("JanuaryCustomersIncome_df", "JanuaryCustomersYtD", spark)
    JanuaryCustomersIncome= spark.sql("SELECT * FROM JanuaryCustomersIncome_df")
    JanuaryCustomersIncome.createOrReplaceTempView("JanuaryCustomersIncome")   

    # Customers with Efapax (once-off) payments
    calculate_OnceOff_Payments( "JanuaryCustomers_OnceOff_df","JanuaryCustomersYtD", spark)
    JanuaryCustomers_OnceOff= spark.sql("SELECT * FROM JanuaryCustomers_OnceOff_df")
    JanuaryCustomers_OnceOff.createOrReplaceTempView("JanuaryCustomers_OnceOff")
  
    # Exclude low risk and family
    BaseJanuaryCustomers = spark.sql("""
      SELECT y.*
      FROM JanuaryCustomersYtD y
      LEFT ANTI JOIN JanuaryCustomersLowRisk l ON y.TaxNumber = l.TaxNumber
      LEFT ANTI JOIN JanuaryCustomersIncome f ON y.TaxNumber = f.TaxNumber
    """)

    # Include OnceOff (even if they are in low risk or family)
    OnceOffCustomersJanuary = spark.sql("""
      SELECT y.*
      FROM JanuaryCustomersYtD y
      INNER JOIN JanuaryCustomers_OnceOff o ON y.TaxNumber = o.TaxNumber
    """)

    # Combine both sets, remove duplicates
    JanuaryCustomers = BaseJanuaryCustomers.unionByName(OnceOffCustomersJanuary).dropDuplicates(["TaxNumber"])
    JanuaryCustomers.createOrReplaceTempView("JanuaryCustomers")


    find_duplicate_payment_type(JanuaryCustomers,DateFrom, DateTo, "DuplicatePaymentFlagsJanuary", spark)

    DuplicatePaymentTypeJanuary = spark.sql("SELECT DISTINCT TaxNumber FROM DuplicatePaymentFlagsJanuary")
    DuplicatePaymentTypeJanuary.createOrReplaceTempView("DuplicatePaymentTypeJanuary")

    FinalRule23 = spark.sql(f"""                                           
                              SELECT	DISTINCT  
                            '23'
                            , a.TaxNumber
                            , a.PaymentAmount
                            , NumOfPayment
                            , NumOfPortfolio
                            , CASE WHEN dp.TaxNumber IS NULL THEN b.PaymentType ELSE NULL END AS PaymentType
                            , b.PortfolioName
                        FROM JanuaryCustomers a
                        JOIN amlPerimeter	b
                        ON a.TaxNumber = b.TaxNumber
                        JOIN AmlExecutionCalendar 
                        ON PayDateRd between DateFrom and DateTo    
                        LEFT JOIN DuplicatePaymentTypeJanuary dp
                        ON dp.TaxNumber = a.TaxNumber  
                                          """)
    FinalRule23.createOrReplaceTempView("FinalRule23")


# Other Months (Except for January)

### Create Initial Perimeters (Existing and New Customers)

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:

	## Find customers that already exist in previous Months in the final AML table  
	ExistInPrevMonths = spark.sql(f"""
								SELECT DISTINCT AmlExecutionCalendar.id, Tax_Number, account_number, payment_amount
                                FROM aml_report_view	
                                join AmlExecutionCalendar on date_from between AMLYear AND PreviousMonthLastDate	
								""")	
	ExistInPrevMonths.createOrReplaceTempView("ExistInPrevMonths")
 
	## Fine new customers (not applicable for the Delta rules - just need to be over 15k & Low Risk < 50k & Declared Income Condition)
	## No need to add date filter here, we should count all payments throughout the year since the customer will be added to AML for the first time
	NewCustomersYtDPerimeter =  spark.sql(f"""
 									SELECT a.TaxNumber 
										, a.PaymentAmount
										, a.PortfolioName
										, a.paydaterd
										from amlPerimeter a
										LEFT JOIN ExistInPrevMonths b on a.TaxNumber = b.Tax_Number  
										where b.Tax_Number IS NULL
								""")
	NewCustomersYtDPerimeter.createOrReplaceTempView("NewCustomersYtDPerimeter")

	# Min and Max PayDateRD for each New Customer
	calculate_YtD_Payments(NewCustomersYtDPerimeter,AMLYear, DateTo, "NewCustomersYtDPerimeter_min_max_sum",spark)
	NewCustomersYtD= spark.sql("SELECT * FROM NewCustomersYtDPerimeter_min_max_sum")
	NewCustomersYtD.createOrReplaceTempView("NewCustomersYtD")
 
	# Low risk Customers with less than 50k sum of payments for New Customers
	calculate_LowRisk_Customers("NewCustomersLowRisk_df", "NewCustomersYtD", spark)
	NewCustomersLowRisk= spark.sql("SELECT * FROM NewCustomersLowRisk_df")
	NewCustomersLowRisk.createOrReplaceTempView("NewCustomersLowRisk")
	
	
 	# Customers whose Payments > than 35% of their Family Income for New Customers
	calculate_FamilyIncome_35percent_rule("NewCustomersFamily_df", "NewCustomersYtD", spark)
	NewCustomersFamily= spark.sql("SELECT * FROM NewCustomersFamily_df")
	NewCustomersFamily.createOrReplaceTempView("NewCustomersFamily")
 
 
	# Customers with Efapax (once-off) payments
	calculate_OnceOff_Payments( "NewCustomers_OnceOff_df","NewCustomersYtD", spark)
	NewCustomers_OnceOff= spark.sql("SELECT * FROM NewCustomers_OnceOff_df")
	NewCustomers_OnceOff.createOrReplaceTempView("NewCustomers_OnceOff")
 
 	# Exclude low risk and family
	BaseNewCustomers = spark.sql("""
    SELECT y.*
    FROM NewCustomersYtD y
    LEFT ANTI JOIN NewCustomersLowRisk l ON y.TaxNumber = l.TaxNumber
    LEFT ANTI JOIN NewCustomersFamily f ON y.TaxNumber = f.TaxNumber
	""")

	# Include OnceOff (even if they are in low risk or family)
	OnceOffCustomers = spark.sql("""
    SELECT y.*
    FROM NewCustomersYtD y
    INNER JOIN NewCustomers_OnceOff o ON y.TaxNumber = o.TaxNumber
	""")

	# Combine both sets, remove duplicates
	NewCustomers = BaseNewCustomers.unionByName(OnceOffCustomers).dropDuplicates(["TaxNumber"])
	NewCustomers.createOrReplaceTempView("NewCustomers")
 

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:

 	## Do the same for the existing customers (the ones that have been present in the AML table in previous months throughout the year)
	
	ExistingCustomersYtD = spark.sql(f""" 
												SELECT  DISTINCT a.TaxNumber
            														,	sum(a.PaymentAmount) PaymentAmount
												from amlPerimeter a
												LEFT JOIN ExistInPrevMonths b on a.TaxNumber = b.Tax_Number		
												where b.Tax_Number IS NOT NULL	
												group by TaxNumber 
												HAVING sum(PaymentAmount) >= 15000
            								""")
	ExistingCustomersYtD.createOrReplaceTempView("ExistingCustomersYtD")
 
 	## Find Customers that have made transactions during the current calculation month
	ExistingCustomersCurrentMonth_init = spark.sql(f""" 
												SELECT  DISTINCT a.TaxNumber
            														,	sum(a.PaymentAmount) PaymentAmount
												from amlPerimeter a
            									join AmlExecutionCalendar on PayDateRd   between DateFrom and DateTo 
												LEFT JOIN ExistInPrevMonths b on a.TaxNumber = b.Tax_Number		
												where b.Tax_Number IS NOT NULL	
												group by TaxNumber 
												HAVING sum(PaymentAmount) >= 15000
            								""")
	ExistingCustomersCurrentMonth_init.createOrReplaceTempView("ExistingCustomersCurrentMonth_init")
	
	# Low risk Customers with less than 50k sum of payments
	calculate_LowRisk_Customers("ExistingCustomersCurrentMonth_LowRisk_df", "ExistingCustomersCurrentMonth_init", spark)
	ExistingCustomersCurrentMonth_LowRisk= spark.sql("SELECT * FROM ExistingCustomersCurrentMonth_LowRisk_df")
	ExistingCustomersCurrentMonth_LowRisk.createOrReplaceTempView("ExistingCustomersCurrentMonth_LowRisk")

 	# Customers whose Payments > than 35% of their Family Income
	calculate_FamilyIncome_35percent_rule("ExistingCustomersFamily_df", "ExistingCustomersYtD", spark)
	ExistingCustomersFamily= spark.sql("SELECT * FROM ExistingCustomersFamily_df")
	ExistingCustomersFamily.createOrReplaceTempView("ExistingCustomersFamily")
 
 	# Customers with Efapax (once-off) payments
	calculate_OnceOff_Payments( "ExistingCustomers_OnceOff_df","ExistingCustomersYtD", spark)
	ExistingCustomers_OnceOff= spark.sql("SELECT * FROM ExistingCustomers_OnceOff_df")
	ExistingCustomers_OnceOff.createOrReplaceTempView("ExistingCustomers_OnceOff")
 
 	# Exclude low risk and family
	BaseExistingCustomers = spark.sql("""
    SELECT y.*
    FROM ExistingCustomersYtD y
    LEFT ANTI JOIN ExistingCustomersCurrentMonth_LowRisk l ON y.TaxNumber = l.TaxNumber
    LEFT ANTI JOIN ExistingCustomersFamily f ON y.TaxNumber = f.TaxNumber
	""")

	# Include OnceOff (even if they are in low risk or family)
	OnceOffCustomers = spark.sql("""
    SELECT y.*
    FROM ExistingCustomersYtD y
    INNER JOIN ExistingCustomers_OnceOff o ON y.TaxNumber = o.TaxNumber
	""")

	# Combine both sets, remove duplicates
	# Final Existing Customers Perimeter
	ExistingCustomersCurrentMonth = BaseExistingCustomers.unionByName(OnceOffCustomers).dropDuplicates(["TaxNumber"])
	ExistingCustomersCurrentMonth.createOrReplaceTempView("ExistingCustomersCurrentMonth") 

### Current MtD

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
    ## Current M Ytd: Calculate for all the current Customers the Year to Date Sum of all payments 
    ## Calculation Period: Start of Year till End of Calculation Period (End of Calculation Month)

    # Existing Customers
    calculate_YtD_Payments(amlPerimeter,AMLYear, DateTo, "CurrentMYtd", spark)
    ExistingCustomersYtd= spark.sql(f"""SELECT a.* 
                                    FROM CurrentMYtd a
                                    LEFT JOIN ExistingCustomersCurrentMonth b ON a.TaxNumber = b.TaxNumber
                                    WHERE b.TaxNumber IS NOT NULL 
                                    """)
    ExistingCustomersYtd.createOrReplaceTempView("ExistingCustomersYtd")

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
    ## Previous M Ytd: Calculate for all the current Customers the Year to Date Sum of all payments 
    ## Calculation Period: Start of Year till End of Previous Month
    calculate_YtD_Payments(amlPerimeter,AMLYear, PreviousMonthDateTo, "PreviousMYtd", spark)
    PreviousMCustomersYtd= spark.sql(f"""SELECT a.* 
                                        FROM PreviousMYtd a
                                        LEFT JOIN ExistingCustomersCurrentMonth b ON a.TaxNumber = b.TaxNumber  
                                        WHERE b.TaxNumber IS NOT NULL""")
    PreviousMCustomersYtd.createOrReplaceTempView("PreviousMCustomersYtd")

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
# Standard step logic: €15k first alert, then every €50k step
	AlertConditions = spark.sql(f"""
										SELECT
											currentytd.Taxnumber,
											previousytd.PaymentAmount PreviousYtd,
											currentytd.PaymentAmount CurrentYtd,

											CASE 
												WHEN previousytd.PaymentAmount <15000 AND currentytd.PaymentAmount >=15000 THEN 1
												WHEN (currentytd.PaymentAmount) >= TRY_CAST(((previousytd.PaymentAmount/50000) + 1) as int) * 50000 THEN 1 
											ELSE 0 END AS ReportFlag
										FROM ExistingCustomersYtd currentytd
										join PreviousMCustomersYtd previousytd
										ON currentytd.TaxNumber = previousytd.TaxNumber 
									""")
	AlertConditions.createOrReplaceTempView("AlertConditions")
	
	# Existing customers: new transactions satisfying alert conditions (exclude already reported)
	ExistingCustomersNewTransactionsInserted = spark.sql(f"""	
											select   
												perim.TaxNumber
												, perim.PaymentAmount
												, PortfolioName
												,PayDateRd
											from amlPerimeter perim
											join AmlExecutionCalendar on PayDateRd   between AMLYear and DateTo  
											join AlertConditions a on a.Taxnumber = perim.TaxNumber
											where PortfolioName in (select portfolioname from PortfolioPerimeter)  
											and a.ReportFlag =1
											AND year(paydaterd) = year(AMLYear)
			""")
	ExistingCustomersNewTransactionsInserted.createOrReplaceTempView("ExistingCustomersNewTransactionsInserted")

	## Existing/to be excuded Transactions: All customers existing in the aml table before the current run
	ExistingCustomersExcluded = spark.sql(f"""
												SELECT   aml.Tax_Number
													, perim.PaymentAmount
													, perim.PortfolioName
													, PayDateRd
												FROM aml_report_view aml
												join amlPerimeter perim on aml.Tax_Number = perim.TaxNumber 
												AND PayDateRd  between aml.date_from and aml.date_to
												join AmlExecutionCalendar on Year(PayDateRd) = Year(AMLYear)
												""")	

	ExistingCustomersExcluded.createOrReplaceTempView("ExistingCustomersExcluded")	

	## Find all distinct Payments that have not been inserted in the AML table prior to this run	
	ExistingCustomersSum = spark.sql(f"""
										SELECT 
											23 AS RuleId, 
											taxnumber,
											COUNT(1) AS NumOfPayment, 
											COUNT(DISTINCT PortfolioName) AS NumOfPortfolio,
											SUM(paymentamount) AS PaymentAmount,
											min(paydaterd) minPayDate,
											max(paydaterd) maxPayDate 
										FROM (
											SELECT *
											FROM ExistingCustomersNewTransactionsInserted
											EXCEPT
											SELECT *
											FROM ExistingCustomersExcluded
										) AS r
										GROUP BY taxnumber;

									""" )
	ExistingCustomersSum.createOrReplaceTempView("ExistingCustomersSum")
	

### Existing Customers min/max date and duplicate Payment Types calculation

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
    ## Calculate the minimum and maximum date for payments per Taxnumber
    ## e.g. If a taxnumber was not inserted in M2 (February), 
    ## and now the sum satisfies the hit/match conditions --> therefore the M period is 01/02-end of current month

    # Existing Customers
    calculate_aml_period(ExistingCustomersSum, "FinalCalculationFieldsExisting",spark)
    FinalExistingCustomers= spark.sql("SELECT * FROM FinalCalculationFieldsExisting")
    FinalExistingCustomers.createOrReplaceTempView("FinalExistingCustomers")

    find_duplicate_payment_type(FinalExistingCustomers, DateFrom,  DateTo,"DuplicatePaymentFlags_Existing", spark)
    DuplicatePaymentTypeExistingCustomers= spark.sql("SELECT * FROM DuplicatePaymentFlags_Existing")
    DuplicatePaymentTypeExistingCustomers.createOrReplaceTempView("DuplicatePaymentTypeExistingCustomers")



### New Customers min/max date and duplicate Payment Types calculation

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
    
    #New Customers
    calculate_aml_period(NewCustomers,"FinalCalculationFieldsNew", spark)
    FinalNewCustomers= spark.sql("SELECT * FROM FinalCalculationFieldsNew")
    FinalNewCustomers.createOrReplaceTempView("FinalNewCustomers")

    find_duplicate_payment_type(FinalNewCustomers, AMLYear, DateTo,"DuplicatePaymentFlags_New", spark)
    DuplicatePaymentTypeNewCustomers= spark.sql("SELECT * FROM DuplicatePaymentFlags_New")
    DuplicatePaymentTypeNewCustomers.createOrReplaceTempView("DuplicatePaymentTypeNewCustomers")



### Create final tables

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
    # All Duplicates 
    DuplicatePaymentTypeAllCustomers= spark.sql(f"""SELECT * FROM DuplicatePaymentTypeNewCustomers 
                                                UNION  SELECT * FROM DuplicatePaymentTypeExistingCustomers""")
    DuplicatePaymentTypeAllCustomers.createOrReplaceTempView("DuplicatePaymentTypeAllCustomers")

    ## Include all existing and non existing customers
    ## New Customers without any prior payments + Existing Customers with new payments
    OverallPayments = spark.sql(f"""
                                SELECT Ruleid, TaxNumber, PaymentAmount, NumOfPayment, NumOfPortfolio, PortfolioName, PaymentType, minPaydate, maxPaydate
                                FROM  FinalNewCustomers
                                UNION
                                SELECT Ruleid, TaxNumber, PaymentAmount, NumOfPayment, NumOfPortfolio, PortfolioName,PaymentType, minPaydate,maxPaydate
                                FROM FinalExistingCustomers b        
                                """)

    OverallPayments.createOrReplaceTempView("OverallPayments")

    FinalRule23 = spark.sql(f"""
                                        SELECT DISTINCT  RuleId
                                                , a.TaxNumber
                                                , a.PaymentAmount
                                                , NumOfPayment
                                                , NumOfPortfolio
                                                , CASE WHEN c.TaxNumber IS NOT NULL THEN NULL ELSE a.PaymentType END AS PaymentType
                                                , a.PortfolioName
                                        FROM OverallPayments a
                                        LEFT JOIN DuplicatePaymentTypeAllCustomers c
                                        ON c.TaxNumber = a.TaxNumber 
       
                                                            """)
    FinalRule23.createOrReplaceTempView("FinalRule23")


In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 1:
    # January HR: all HR customers with YtD ≥ €3k qualify (no step comparison)
    calculate_HighRisk_YtD(DateFrom, DateTo, "HR_JanuaryCustomersYtD", spark)

    # Build FinalRule23_HR for January (same schema as FinalRule23)
    FinalRule23_HR = spark.sql(f"""
        SELECT DISTINCT
            23 AS RuleId,
            a.TaxNumber,
            a.PaymentAmount,
            a.NumOfPayment,
            a.NumOfPortfolio,
            CASE WHEN dp.countPaymentType > 1 THEN NULL 
                 ELSE pf.PaymentType END AS PaymentType,
            pf.PortfolioName
        FROM HR_JanuaryCustomersYtD a
        JOIN amlPerimeterFull pf ON pf.TaxNumber = a.TaxNumber AND pf.IsHighRiskExposure = 1
        LEFT JOIN (
            SELECT TaxNumber, COUNT(DISTINCT PaymentType) AS countPaymentType
            FROM amlPerimeterFull
            WHERE IsHighRiskExposure = 1
            GROUP BY TaxNumber
            HAVING COUNT(DISTINCT PaymentType) > 1
        ) dp ON dp.TaxNumber = a.TaxNumber
    """)
    FinalRule23_HR.createOrReplaceTempView("FinalRule23_HR")
    print(f"FinalRule23_HR (January): {FinalRule23_HR.count():,} rows")

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
# HR: Unified step logic — €3k threshold, €3k steps, no exclusions
    
    # 1. Current YtD (AMLYear → DateTo) — HR customers with YtD ≥ €3k
    calculate_HighRisk_YtD(AMLYear, DateTo, "HR_CurrentYtD", spark)
    
    # 2. Previous YtD (AMLYear → PreviousMonthDateTo) — raw sum, no threshold
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW HR_PreviousYtD AS
        SELECT TaxNumber, SUM(PaymentAmount) AS PaymentAmount
        FROM amlPerimeterFull
        WHERE IsHighRiskExposure = 1
          AND PayDateRd BETWEEN '{AMLYear}' AND '{PreviousMonthDateTo}'
        GROUP BY TaxNumber
    """)
    
    # 3. Current month activity filter
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW HR_CurrentPeriodPayments AS
        SELECT DISTINCT TaxNumber
        FROM amlPerimeterFull
        WHERE IsHighRiskExposure = 1
          AND PayDateRd BETWEEN '{DateFrom}' AND '{DateTo}'
    """)
    
    # 4. Step logic: alert if current crosses next €3k step vs previous
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW HR_AlertConditions AS
        SELECT
            c.TaxNumber,
            COALESCE(p.PaymentAmount, 0) AS PreviousYtd,
            c.PaymentAmount AS CurrentYtd,
            CASE 
                WHEN COALESCE(p.PaymentAmount, 0) < 3000 AND c.PaymentAmount >= 3000 THEN 1
                WHEN c.PaymentAmount >= TRY_CAST((COALESCE(p.PaymentAmount, 0) / 3000 + 1) AS INT) * 3000 THEN 1
                ELSE 0
            END AS ReportFlag
        FROM HR_CurrentYtD c
        LEFT JOIN HR_PreviousYtD p ON p.TaxNumber = c.TaxNumber
        JOIN HR_CurrentPeriodPayments cp ON cp.TaxNumber = c.TaxNumber
    """)
    
    # 5. Final HR alerts
    HR_OverallPayments = spark.sql(f"""
        SELECT 
            23 AS RuleId,
            a.TaxNumber,
            a.PaymentAmount,
            a.NumOfPayment,
            a.NumOfPortfolio,
            a.minPayDate,
            a.maxPayDate
        FROM HR_CurrentYtD a
        JOIN HR_AlertConditions ac ON ac.TaxNumber = a.TaxNumber
        WHERE ac.ReportFlag = 1
    """)
    HR_OverallPayments.createOrReplaceTempView("HR_OverallPayments")
    
    print(f"HR alerts triggered: {HR_OverallPayments.count():,} TaxNumbers")

    # Build FinalRule23_HR
    FinalRule23_HR = spark.sql(f"""
        SELECT DISTINCT
            23 AS RuleId,
            a.TaxNumber,
            a.PaymentAmount,
            a.NumOfPayment,
            a.NumOfPortfolio,
            CASE WHEN dp.countPaymentType > 1 THEN NULL 
                 ELSE pf.PaymentType END AS PaymentType,
            pf.PortfolioName
        FROM HR_OverallPayments a
        JOIN amlPerimeterFull pf ON pf.TaxNumber = a.TaxNumber AND pf.IsHighRiskExposure = 1
        LEFT JOIN (
            SELECT TaxNumber, COUNT(DISTINCT PaymentType) AS countPaymentType
            FROM amlPerimeterFull
            WHERE IsHighRiskExposure = 1
            GROUP BY TaxNumber
            HAVING COUNT(DISTINCT PaymentType) > 1
        ) dp ON dp.TaxNumber = a.TaxNumber
    """)
    FinalRule23_HR.createOrReplaceTempView("FinalRule23_HR")
    print(f"FinalRule23_HR (Other Months): {FinalRule23_HR.count():,} rows")

In [0]:
if AMLExecutionFlag == 1:
    # UNION High Risk results with Standard FinalRule23 (add IsHighRisk flag for join filtering)
    FinalRule23_combined = spark.sql("""
        SELECT RuleId, TaxNumber, PaymentAmount, NumOfPayment, NumOfPortfolio, PaymentType, PortfolioName, 0 AS IsHighRisk
        FROM FinalRule23
        UNION
        SELECT RuleId, TaxNumber, PaymentAmount, NumOfPayment, NumOfPortfolio, PaymentType, PortfolioName, 1 AS IsHighRisk
        FROM FinalRule23_HR
    """)
    # Override FinalRule23 view with combined results
    FinalRule23_combined.createOrReplaceTempView("FinalRule23")
    
    hr_count = spark.sql("SELECT COUNT(DISTINCT TaxNumber) FROM FinalRule23_HR").first()[0]
    print(f"FinalRule23 combined: {FinalRule23_combined.count():,} rows (HR: {hr_count} unique TaxNumbers)")

    # Add HR_OverallPayments alongside Standard (as separate view to avoid corrupting FinalRule23 view chain)
    if JanuaryFlag == 0:
        # Keep OverallPayments UNTOUCHED (FinalRule23 Standard view depends on it!)
        # Create combined view under a DIFFERENT name for FinalTableRank date lookup
        spark.sql("""
            CREATE OR REPLACE TEMP VIEW OverallPayments_All AS
            SELECT * FROM OverallPayments
            UNION
            SELECT RuleId, TaxNumber, PaymentAmount, NumOfPayment, NumOfPortfolio, 
                   NULL AS PortfolioName, NULL AS PaymentType, minPayDate, maxPayDate
            FROM HR_OverallPayments
        """)

    # Restore amlPerimeter to full (needed for downstream update_portfolio_of_payment / update_cdi)
    spark.sql("CREATE OR REPLACE TEMP VIEW amlPerimeter AS SELECT * FROM amlPerimeterFull")
    print("amlPerimeter restored to full (HR + Standard) for downstream joins")

In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 0:
	FinalTableRank_withchecksum = spark.sql(f"""
							select	distinct NULL id
									,current_date() Report_Date
									,n.minPayDate DateFrom
									,n.maxPayDate DateTo	
									,a.exposurekey as AccountNumber
									,a.CDI 
									,a.taxnumber Tax_Number
									,ltrim(rtrim(concat(a.firstname, ' ', a.lastname, ' ', a.fathername))) Customer_Name
									,a.legalentity Legal_Entity
									,a.Nationality
									,CASE WHEN a.legalentity = 'No' THEN  TRY_CAST(a.birthdate as date) ELSE null END Birth_Date
									,a.Occupation 
									,f.PortfolioName portfolio_payment
									,f.PaymentAmount payment_amount
									, f.paymenttype  PaymentType
									,f.numofpayment num_of_payments
									,CASE WHEN comm.TaxNumber is not null THEN comm.PortfolioName ELSE f.PortfolioName END Customer_Portfolio
									,'23' rule_id
									,'M'  rule_type
									,CASE WHEN a.IsHighRiskExposure = 1 
										 THEN 'High Risk - YtD sum >= 3.000 per TAX number (HR party involved)'
										 ELSE 'YtD sum of installments/ payment >=15.000 per unique TAX number across all NOT BANKING portfolios (not per λογαριασμό)' 
									 END AS rule_name
									,a.aml_sch_id aml_sch_id
									,a.GroupName
									,a.DeceasedFlag
									,a.PartyStatus
									,a.PartyStatusDate
									,a.PayerCDI
									,a.PaymentFileChannel
									,tp.total_payments_ever
									,tp.total_payments_current_year
									,a.LegitimizationExpiredDate
									,ROW_NUMBER ( )   
								OVER (partition by a.taxnumber, f.paymentamount order by a.CDI ) rnk
									,crc32(concat_ws("|", a.cdi, a.aml_sch_id,'23'))  checksum

							from FinalRule23 f 
							join amlPerimeter a on a.TaxNumber = f.TaxNumber AND a.IsHighRiskExposure = f.IsHighRisk	
							left join amlCommonCustomers comm on comm.TaxNumber=f.TaxNumber
							left join OverallPayments_All n on n.taxnumber = a.taxnumber		
							LEFT JOIN TotalPaymentsPerimeter tp ON tp.TaxNumber = a.TaxNumber
	""")

	FinalTableRank_withchecksum.createOrReplaceTempView("FinalTableRank_withchecksum")


In [0]:
if AMLExecutionFlag == 1 and JanuaryFlag == 1:
	FinalTableRank_withchecksum = spark.sql(f"""
							select	distinct 
							TRY_CAST(NULL as int) id
									,current_date() Report_Date
									,amlcheck.DateFrom
									,amlcheck.DateTo	
									,a.exposurekey as AccountNumber
									,a.CDI 
									,a.taxnumber Tax_Number
									,ltrim(rtrim(concat(a.firstname, ' ', a.lastname, ' ', a.fathername))) Customer_Name
									,a.legalentity Legal_Entity
									,a.Nationality
									,CASE WHEN a.legalentity = 'No' THEN  TRY_CAST(a.birthdate as date) ELSE null END Birth_Date
									,a.Occupation 
									,f.PortfolioName portfolio_payment
									,f.PaymentAmount payment_amount
									, f.paymenttype  PaymentType
									,f.numofpayment num_of_payments
									,CASE WHEN comm.TaxNumber is not null THEN comm.PortfolioName ELSE f.PortfolioName END Customer_Portfolio
									,'23' rule_id
									,'M'  rule_type
									,CASE WHEN a.IsHighRiskExposure = 1 
										 THEN 'High Risk - YtD sum >= 3.000 per TAX number (HR party involved)'
										 ELSE 'YtD sum of installments/ payment >=15.000 per unique TAX number across all NOT BANKING portfolios (not per λογαριασμό)' 
									 END AS rule_name
									,amlcheck.aml_sch_id aml_sch_id
									,a.GroupName
									,a.DeceasedFlag
									,a.PartyStatus
									,a.PartyStatusDate
									,a.PayerCDI
									,a.PaymentFileChannel
									,tp.total_payments_ever
									,tp.total_payments_current_year
									,a.LegitimizationExpiredDate
									,ROW_NUMBER ( )   
								OVER (partition by a.taxnumber, f.paymentamount order by a.CDI ) rnk
									,crc32(concat_ws("|", a.cdi, amlcheck.aml_sch_id,'23'))  checksum
							from FinalRule23 f 
								join amlPerimeter a on a.TaxNumber = f.TaxNumber AND a.IsHighRiskExposure = f.IsHighRisk	
								join AmlExecutionCalendar amlcheck on PayDateRd between AMLYear and DateTo
								left join amlCommonCustomers comm on comm.TaxNumber=f.TaxNumber
					LEFT JOIN TotalPaymentsPerimeter tp ON tp.TaxNumber = a.TaxNumber
	""")

	FinalTableRank_withchecksum.createOrReplaceTempView("FinalTableRank_withchecksum")


In [0]:
if AMLExecutionFlag ==1:

	update_portfolio_of_payment("CustomerPortfolioAggtemp")
	update_cdi("CustomerCDIAggtemp")

	amlReportRunRank = spark.sql(f"""SELECT DISTINCT 
                          TRY_CAST(id as int) as id
												, Report_Date 
												, DateFrom date_from
												, DateTo date_to
												, COALESCE(c.CDI, f.CDI) AS CDI
												, Tax_Number
												, Customer_Name
												, Legal_Entity
												, Nationality
											  , TRY_CAST(Birth_Date AS DATE) AS Birth_Date
												, Occupation 
												, COALESCE(s.portfolio, f.portfolio_payment) AS portfolio_payment
												, TRY_CAST(payment_amount AS DECIMAL(18,2)) AS payment_amount
												, TRY_CAST(num_of_payments AS INT) AS num_of_payments
												, COALESCE(s.portfolio, f.portfolio_payment) AS Customer_Portfolio 
												, TRY_CAST(rule_id AS INT) AS rule_id
												, rule_type
												, rule_name
												, TRY_CAST(aml_sch_id AS INT) AS aml_sch_id
												, checksum AS chsum
												, accountnumber account_number
												, PaymentType payment_type
												, f.GroupName
												, f.DeceasedFlag
												, f.PartyStatus
												, f.PartyStatusDate
												, f.PayerCDI
												, f.PaymentFileChannel
												, TRY_CAST(f.total_payments_ever AS DECIMAL(18,2))          AS total_payments_ever
									, TRY_CAST(f.total_payments_current_year AS DECIMAL(18,2))  AS total_payments_current_year
										, TRY_CAST(f.LegitimizationExpiredDate AS DATE)             AS LegitimizationExpiredDate
				FROM FinalTableRank_withchecksum f
								LEFT JOIN CustomerPortfolioAggtemp s
								ON f.Tax_number = s.taxnumber
								LEFT JOIN CustomerCDIAggtemp c
								ON f.Tax_number = c.taxnumber    	
								WHERE rnk = 1""")
	amlReportRunRank.createOrReplaceTempView("amlReportRunRank")
								


In [0]:
if AMLExecutionFlag == 1:
    # Get the max ID from the existing table (default to 0 if empty)
    max_id = aml_report_view.agg({"id": "max"}).first()[0] or 0


    w = Window.orderBy("Tax_Number", "CDI")
    amlReportRun_withid = (
        amlReportRunRank
        .withColumn("id", (lit(max_id) + row_number().over(w)).cast("int"))
    )

    amlReportRun_withid.cache()
    print(f"amlReportRun_withid: {amlReportRun_withid.count():,} rows (cached)")
    amlReportRun_withid.createOrReplaceTempView("amlReportRun_withid")


In [0]:
if AMLExecutionFlag == 1:
	amlReportRun = spark.sql(f"""
							SELECT DISTINCT *
							FROM amlReportRun_withid
							UNION
      						SELECT TRY_CAST(id as int) AS id
											, Report_Date
											, date_from
											, date_to
											, CDI
											, Tax_Number
											, Customer_Name
											, Legal_Entity
											, Nationality
											, TRY_CAST(Birth_Date AS DATE) AS Birth_Date
											, Occupation 
											, portfolio_payment
											, TRY_CAST(payment_amount AS DECIMAL(18,2)) AS payment_amount
											, TRY_CAST(num_of_payments AS INT) AS num_of_payments
											, Customer_Portfolio 
											, TRY_CAST(rule_id AS INT) AS rule_id
											, rule_type
											, rule_name
											, TRY_CAST(aml_sch_id as int) AS aml_sch_id
											, chsum AS chsum  
											, account_number
											, payment_type
											, NULL AS GroupName
											, NULL AS DeceasedFlag
											, NULL AS PartyStatus
											, NULL AS PartyStatusDate
											, NULL AS PayerCDI
											, NULL AS PaymentFileChannel
									, NULL AS total_payments_ever
									, NULL AS total_payments_current_year
									, NULL AS LegitimizationExpiredDate
                            FROM aml_report_view h
""") 

	amlReportRun.createOrReplaceTempView("amlReportRun")


## Append to final Calendar and AML Tables

In [0]:
if AMLExecutionFlag == 1:
  amlReportRun.write.mode("overwrite").saveAsTable(OUTPUT_TABLE)
  print(f"Written {amlReportRun.count():,} rows to {OUTPUT_TABLE}")

  # Update execution calendar to mark this period as executed
  if not _force_rerun:
      spark.sql(f"""
          UPDATE `{env}-cdh-delivery`.aml.execution_calendar
          SET ActualExecution = current_date()
          WHERE DateFrom = '{DateFrom}' AND DateTo = '{DateTo}'
            AND (ActualExecution IS NULL OR ActualExecution = 'NULL')
      """)
      print("Execution calendar updated")



In [0]:
if AMLExecutionFlag == 1:
    # NULL cleanup on output table
    spark.sql(f"UPDATE {OUTPUT_TABLE} SET Birth_Date = NULL WHERE CAST(Birth_Date AS STRING) = 'NULL'")
    spark.sql(f"UPDATE {OUTPUT_TABLE} SET record_status = NULL WHERE record_status = 'NULL'")
    spark.sql(f"UPDATE {OUTPUT_TABLE} SET account_number = NULL WHERE account_number = 'NULL'")
    spark.sql(f"UPDATE {OUTPUT_TABLE} SET payment_type = NULL WHERE payment_type = 'NULL'")
    print(f"NULL cleanup completed on {OUTPUT_TABLE}") 